In [7]:
import numpy as np
import pandas as pd

# -----------------------------
# 1) Labeled example dataset
# -----------------------------
df = pd.DataFrame({
    "user": ["A", "A", "B", "B", "C", "C"],
    "item": ["Movie1", "Movie2", "Movie1", "Movie3", "Movie2", "Movie3"],
    "rating": [5, 3, 4, 2, 4, 1]
})

print("All observed ratings:")
print(df)

# -----------------------------
# 2) Train MF with SGD on TRAIN split only
# -----------------------------
def train_mf_sgd(train_df, user_ids, item_ids, k, lr=0.05, reg=0.01, epochs=1200, seed=42):
    rng = np.random.default_rng(seed)
    n_users = len(user_ids)
    n_items = len(item_ids)
    U = rng.normal(0, 0.1, (n_users, k))
    V = rng.normal(0, 0.1, (n_items, k))

    for _ in range(epochs):
        # shuffle rows each epoch
        shuffled = train_df.sample(frac=1.0, random_state=int(rng.integers(0, 1_000_000)))
        for row in shuffled.itertuples(index=False):
            u = user_ids[row.user]
            i = item_ids[row.item]
            r = float(row.rating)

            pred = float(U[u].dot(V[i]))
            err = r - pred

            # SGD updates with L2 regularization
            u_old = U[u].copy()
            U[u] += lr * (err * V[i] - reg * U[u])
            V[i] += lr * (err * u_old - reg * V[i])

    return U, V

def rmse(eval_df, U, V, user_ids, item_ids):
    se = 0.0
    n = 0
    for row in eval_df.itertuples(index=False):
        u = user_ids[row.user]
        i = item_ids[row.item]
        r = float(row.rating)
        pred = float(U[u].dot(V[i]))
        se += (r - pred) ** 2
        n += 1
    return float(np.sqrt(se / max(n, 1)))

# -----------------------------
# 3) Make a train/val split (hold out 2 ratings)
#    (Tiny data => noisy; still illustrates the idea)
# -----------------------------
val_idx = [1, 5]  # hold out (A, Movie2)=3 and (C, Movie3)=1
val_df = df.iloc[val_idx].reset_index(drop=True)
train_df = df.drop(index=val_idx).reset_index(drop=True)

print("\nTrain split:")
print(train_df)
print("\nValidation split (held out):")
print(val_df)

# -----------------------------
# 4) Map IDs (based on ALL users/items so val can be evaluated)
# -----------------------------
user_ids = {u: i for i, u in enumerate(df["user"].unique())}
item_ids = {m: i for i, m in enumerate(df["item"].unique())}


# ----------------------------

All observed ratings:
  user    item  rating
0    A  Movie1       5
1    A  Movie2       3
2    B  Movie1       4
3    B  Movie3       2
4    C  Movie2       4
5    C  Movie3       1

Train split:
  user    item  rating
0    A  Movie1       5
1    B  Movie1       4
2    B  Movie3       2
3    C  Movie2       4

Validation split (held out):
  user    item  rating
0    A  Movie2       3
1    C  Movie3       1


In [ ]:
# -----------------------------
# 5) Sweep k and show train vs val RMSE
# -----------------------------
ks = [] #fill in with k values
rows = []
for k in ks:
    U, V = train_mf_sgd(train_df, user_ids, item_ids, k=k, lr=0.05, reg=0.01, epochs=1200, seed=42)
    rows.append({
        "k": k,
        "train_RMSE": rmse(train_df, U, V, user_ids, item_ids),
        "val_RMSE": rmse(val_df, U, V, user_ids, item_ids),
    })

results = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
print(results)
